In [ ]:
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input, GRU, Conv1D, MaxPool1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score, mean_squared_error, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, cross_val_score, TimeSeriesSplit
from tensorflow.keras.losses import Huber
from xgboost import XGBRegressor
import tensorflow as tf
import pandas as pd
import numpy as np
import pickle

In [ ]:
df = pd.read_csv("NIFTY.csv")
print(df.shape)
df.head()

(4977, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,NaN,^CNX100,^CNX100,^CNX100,^CNX100,^CNX100,NaN,NaN,NaN,NaN
1,2005-11-30,2605.10009765625,2605.110107421875,2605.10009765625,2605.110107421875,0,^CNX100,NaN,NaN,NaN
2,2005-12-01,2649.75,2654.10009765625,2595.449951171875,2614.64990234375,0,^CNX100,1.713942,NaN,NaN
3,2005-12-02,2652.300048828125,2679.89990234375,2646.89990234375,2670.35009765625,0,^CNX100,0.096237,NaN,NaN
4,2005-12-05,2619.699951171875,2663.550048828125,2614.050048828125,2663.550048828125,0,^CNX100,-1.229126,NaN,NaN


In [ ]:
df = df.iloc[1:4977,:]
print(df.shape)
df.head()

(4976, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
1,2005-11-30,2605.10009765625,2605.110107421875,2605.10009765625,2605.110107421875,0,^CNX100,NaN,NaN,NaN
2,2005-12-01,2649.75,2654.10009765625,2595.449951171875,2614.64990234375,0,^CNX100,1.713942,NaN,NaN
3,2005-12-02,2652.300048828125,2679.89990234375,2646.89990234375,2670.35009765625,0,^CNX100,0.096237,NaN,NaN
4,2005-12-05,2619.699951171875,2663.550048828125,2614.050048828125,2663.550048828125,0,^CNX100,-1.229126,NaN,NaN
5,2005-12-06,2618.60009765625,2646.449951171875,2604.050048828125,2622.300048828125,0,^CNX100,-0.041984,NaN,NaN


In [ ]:
df.isna().sum()

Date                0
Close               0
High                0
Low                 0
Open                0
Volume              0
Ticker              0
Daily_Return_%      1
MA_50              49
MA_200            199
dtype: int64

In [ ]:
colist = ['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return_%']
for col in colist:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"])
for col in colist:
    print(df[col].dtype)

float64
float64
float64
float64
int64
float64


#Feature Engineering

In [ ]:
for col in df.columns:
    print(col, "->",df[col].dtype,)

Date -> datetime64[us]
Close -> float64
High -> float64
Low -> float64
Open -> float64
Volume -> int64
Ticker -> str
Daily_Return_% -> float64
MA_50 -> float64
MA_200 -> float64


In [ ]:
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
df.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,2005-11-30,2605.100098,2605.110107,2605.100098,2605.110107,0,^CNX100,NaN,NaN,NaN
1,2005-12-01,2649.750000,2654.100098,2595.449951,2614.649902,0,^CNX100,1.713942,NaN,NaN
2,2005-12-02,2652.300049,2679.899902,2646.899902,2670.350098,0,^CNX100,0.096237,NaN,NaN
3,2005-12-05,2619.699951,2663.550049,2614.050049,2663.550049,0,^CNX100,-1.229126,NaN,NaN
4,2005-12-06,2618.600098,2646.449951,2604.050049,2622.300049,0,^CNX100,-0.041984,NaN,NaN


In [ ]:
#Basic Columns
def preporcessing(df):
    df["NIFTY_100"] = df["Close"].pct_change()
    df["Volatility_100"] = (df["NIFTY_100"].rolling(20).std())

    prev_close = df["Close"].shift(-1)
    df["TR"] = pd.concat(
        [
            df["High"] - df["Close"], 
            (df["High"] - prev_close).abs(), 
            (df["Low"] - prev_close).abs()
        ], axis=1
    ).max(axis=1)

    #Rolling volume
    df["Rolling Volume"] = df.groupby("Ticker")["Volume"].transform(lambda x : x.rolling(20).mean())

    #Rolling return
    df["Rolling_Return_5D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(5))
    df["Rolling_Return_10D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(10))
    df["Rolling_Return_20D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(20))

    #Ranges
    df["High_low_%"] = ((df["High"] - df["Low"])/df["Close"])*100
    df["Open_close_%"] = ((df["Close"] - df["Open"]).abs()/df["Close"])*100

    #Lagged Volatility
    df["Rolling_Volatility_20D"] = df.groupby("Ticker")["Daily_Return_%"].transform(lambda x: x.rolling(20).std())
    df["Lagged_Volatility_1"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(1))
    df["Lagged_Volatility_5"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(5))
    df["Lagged_Volatility_10"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(10))

    #MA_to_Price
    df["MA50_to_Price"] = (df["MA_50"]/df["Close"])
    df["MA200_to_Price"] = (df["MA_200"]/df["Close"])

    #Volatility
    df["Daily_Return"] = df.groupby("Ticker")["Close"].pct_change()

    df["Rolling_Volatility_5D"] = df.groupby("Ticker")["Daily_Return"].transform(lambda x: x.rolling(5).std())
    df["Rolling_Volatility_10D"] = df.groupby("Ticker")["Daily_Return"].transform(lambda x: x.rolling(10).std())

    def future_shift(col):
        return col.iloc[::-1].rolling(5).std().iloc[::-1].shift(-1)

    #Final prediction that is to be made
    df["future_shift_5"] = df.groupby("Ticker")["Daily_Return"].transform(future_shift)

In [ ]:
print(df[["Date", "Daily_Return_%", "future_shift_5"]].tail(10))

           Date  Daily_Return_%  future_shift_5
4966 2026-02-09        0.724409        0.008416
4967 2026-02-10        0.242749        0.008399
4968 2026-02-11        0.151405        0.008698
4969 2026-02-12       -0.542450        0.010744
4970 2026-02-13       -1.344713        0.009155
4971 2026-02-16        0.854460             NaN
4972 2026-02-17        0.227054             NaN
4973 2026-02-18        0.415164             NaN
4974 2026-02-19       -1.487281             NaN
4975 2026-02-20        0.485236             NaN


In [ ]:
df = df.dropna()
df.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200,...,Rolling_Volatility_20D,Lagged_Volatility_1,Lagged_Volatility_5,Lagged_Volatility_10,MA50_to_Price,MA200_to_Price,Daily_Return,Rolling_Volatility_5D,Rolling_Volatility_10D,future_shift_5
199,2006-09-19,3348.350098,3403.899902,3331.300049,3394.800049,0,^CNX100,-1.047637,3153.158994,3058.710750,...,1.065387,1.028316,0.980829,0.697964,0.941705,0.913498,-0.010476,0.010730,0.013598,0.009519
200,2006-09-20,3388.800049,3394.350098,3314.100098,3318.750000,0,^CNX100,1.208056,3161.117993,3062.629249,...,1.053750,1.065387,1.028683,0.623630,0.932813,0.903750,0.012081,0.008237,0.014144,0.008916
201,2006-09-21,3436.300049,3439.800049,3413.500000,3413.550049,0,^CNX100,1.401676,3168.625996,3066.562000,...,1.072433,1.053750,1.029501,0.625134,0.922104,0.892402,0.014017,0.009742,0.014483,0.007153
202,2006-09-22,3427.949951,3445.550049,3410.750000,3414.649902,0,^CNX100,-0.242997,3176.421997,3070.440249,...,1.076358,1.072433,1.029120,0.626083,0.926624,0.895707,-0.002430,0.010185,0.014550,0.006808
203,2006-09-25,3407.300049,3435.649902,3400.050049,3435.250000,0,^CNX100,-0.602398,3184.655000,3074.378250,...,1.091045,1.076358,1.028316,0.999139,0.934656,0.902292,-0.006024,0.011001,0.009316,0.006344


#Training and Testing

In [11]:
drop_cols = ["NIFTY_100", "Rolling_Volatility_20D"]
X = df.drop(["future_shift_5", "Ticker", "Date", "Daily_Return"]+drop_cols, axis = 1)
Y = df["future_shift_5"]

mxscale = MinMaxScaler()
X_scaled = mxscale.fit_transform(X)

Xtrain, Xtest, Ytrain, Ytest = train_test_split(X_scaled, Y, test_size=0.2, random_state = 42)

print(Xtrain.shape)
print(Xtest.shape)
print(Ytrain.shape)
print(Ytest.shape)

(3817, 23)
(955, 23)
(3817,)
(955,)


In [35]:
lightgbm = LGBMRegressor(
    n_estimators=1000, max_depth=20, random_state=42, min_child_samples=10
)

tscv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

score1 = cross_val_score(
    lightgbm,
    Xtrain, 
    Ytrain,
    cv=tscv,
    scoring="r2",
    n_jobs=-1
)

lightgbm.fit(Xtrain, Ytrain)

xgb = XGBRegressor(n_estimators=1000, random_state=42)

tscv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

score2 = cross_val_score(
    xgb,
    Xtrain, 
    Ytrain,
    cv=tscv,
    scoring='r2'
)

xgb.fit(Xtrain, Ytrain)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000833 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5865
[LightGBM] [Info] Number of data points in the train set: 3817, number of used features: 23
[LightGBM] [Info] Start training from score 0.010300


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [36]:
YXpred = xgb.predict(Xtest)
print("r2_score", r2_score(Ytest, YXpred))
print("mse", mean_squared_error(Ytest, YXpred))
print("mae", mean_absolute_error(Ytest, YXpred))

print(score1)
print(score1.mean())

r2_score 0.6805470672551927
mse 1.463787664219514e-05
mae 0.0025509823548837017
[0.79502625 0.74767113 0.74859761 0.7038609  0.72768481]
0.7445681412718297


In [37]:
YLpred = lightgbm.predict(Xtest)
print("mse",mean_squared_error(Ytest, YLpred))
print("mae",mean_absolute_error(Ytest, YLpred))
print("r2score",r2_score(Ytest, YLpred)*100)

print(score2)
print(score2.mean())

mse 1.2468159496994991e-05
mae 0.002401285509396495
r2score 72.78983684174722
[0.76235121 0.73539782 0.71557359 0.70037633 0.67640514]
0.7180208151530301


In [ ]:
pickle.dump("model.pkl", lightgbm)

In [46]:
Ypred = (0.7*YLpred + 0.3*YXpred)
print("mse", mean_squared_error(Ytest, Ypred))
print("mae", mean_absolute_error(Ytest, Ypred))
print("r2_score", r2_score(Ytest, Ypred))

mse 1.2009395732327341e-05
mae 0.0023810443364150713
r2_score 0.7379103007244896


In [32]:
X.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return_%', 'MA_50',
       'MA_200', 'Volatility_100', 'TR', 'Rolling Volume', 'Rolling_Return_5D',
       'Rolling_Return_10D', 'Rolling_Return_20D', 'High_low_%',
       'Open_close_%', 'Lagged_Volatility_1', 'Lagged_Volatility_5',
       'Lagged_Volatility_10', 'MA50_to_Price', 'MA200_to_Price',
       'Rolling_Volatility_5D', 'Rolling_Volatility_10D'],
      dtype='str')

#Improving the model

In [33]:
importance = pd.Series(
    lightgbm.feature_importances_,
    index = X.columns
).sort_values(ascending=False)

print(importance)

TR                        2007
Daily_Return_%            1832
Rolling_Volatility_5D     1800
Rolling_Return_5D         1784
Rolling_Return_10D        1775
Open_close_%              1718
High_low_%                1681
Rolling_Return_20D        1658
MA200_to_Price            1652
MA50_to_Price             1638
Rolling_Volatility_10D    1528
Volume                    1515
Rolling Volume            1476
Lagged_Volatility_10      1444
Lagged_Volatility_5       1263
Volatility_100            1083
Lagged_Volatility_1       1042
MA_200                     748
MA_50                      732
Close                      673
High                       333
Open                       310
Low                        308
dtype: int32


#Using LSTM

In [37]:
dfcopy = df
cols = ['Close', 'High', 'Low', 'Open', 'Volume',
       'Daily_Return_%', 'MA_50', 'MA_200', 'Volatility_100',
       'TR', 'Rolling Volume', 'Rolling_Return_5D', 'Rolling_Return_10D',
       'Rolling_Return_20D', 'High_low_%', 'Open_close_%',
       'Lagged_Volatility_1', 'Lagged_Volatility_5', 'Lagged_Volatility_10',
       'MA50_to_Price', 'MA200_to_Price', 'Rolling_Volatility_5D', 'Rolling_Volatility_10D']

In [38]:
dfcopy.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200,...,Rolling_Volatility_20D,Lagged_Volatility_1,Lagged_Volatility_5,Lagged_Volatility_10,MA50_to_Price,MA200_to_Price,Daily_Return,Rolling_Volatility_5D,Rolling_Volatility_10D,future_shift_5
199,2006-09-19,3348.350098,3403.899902,3331.300049,3394.800049,0,^CNX100,-1.047637,3153.158994,3058.710750,...,1.065387,1.028316,0.980829,0.697964,0.941705,0.913498,-0.010476,0.010730,0.013598,0.009519
200,2006-09-20,3388.800049,3394.350098,3314.100098,3318.750000,0,^CNX100,1.208056,3161.117993,3062.629249,...,1.053750,1.065387,1.028683,0.623630,0.932813,0.903750,0.012081,0.008237,0.014144,0.008916
201,2006-09-21,3436.300049,3439.800049,3413.500000,3413.550049,0,^CNX100,1.401676,3168.625996,3066.562000,...,1.072433,1.053750,1.029501,0.625134,0.922104,0.892402,0.014017,0.009742,0.014483,0.007153
202,2006-09-22,3427.949951,3445.550049,3410.750000,3414.649902,0,^CNX100,-0.242997,3176.421997,3070.440249,...,1.076358,1.072433,1.029120,0.626083,0.926624,0.895707,-0.002430,0.010185,0.014550,0.006808
203,2006-09-25,3407.300049,3435.649902,3400.050049,3435.250000,0,^CNX100,-0.602398,3184.655000,3074.378250,...,1.091045,1.076358,1.028316,0.999139,0.934656,0.902292,-0.006024,0.011001,0.009316,0.006344


In [39]:
for ticker, data in dfcopy.groupby("Ticker"):
    Xsc = []
    Ysc = []

    X = data[cols].values
    Y = data["future_shift_5"]
    msc = MinMaxScaler()
    Xmsc = msc.fit_transform(X)

    window = 60
    for i in range(window, len(X)):
        Xsc.append(Xmsc[i-window:i])
        Ysc.append(Y.iloc[i])

    Xsc = np.array(Xsc)
    Ysc = np.array(Ysc)

In [40]:
Xsc.dtype

dtype('float64')

In [41]:
split = int(len(Xsc)*0.8)

XLTrain = Xsc[:split]
XLTest = Xsc[split:]

YLTrain = Ysc[:split]
YLTest = Ysc[split:]

In [42]:
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(
    YLTrain.reshape(-1,1)
)

yt_scaled = y_scaler.transform(
    YLTest.reshape(-1,1)
)

In [43]:
XLTrain.dtype
YLTrain.dtype

dtype('float64')

In [44]:
print("XLTrain", XLTrain.shape)
print("XLTest", XLTest.shape)
print("YLTrain", YLTrain.shape)
print("YLTest", YLTest.shape)

XLTrain (3769, 60, 23)
XLTest (943, 60, 23)
YLTrain (3769,)
YLTest (943,)


In [65]:
modelL = Sequential(
    [Input(shape=(60, XLTrain.shape[2])),
    GRU(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)]
)

modelL.compile(
    optimizer=Adam(learning_rate=0.05),
    loss=Huber()
)

call_back = [EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)
    # ReduceLROnPlateau(
    #     moniter='val_loss',
    #     patience=5,
    #     factor=0.5,
    #     min_LR=1e-6
    # )
]
modelL.fit(
    XLTrain, 
    y_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=call_back
)

# modelC = Sequential([
#     Input(shape=(60, XLTrain.shape[2])),
#     Conv1D(filters=64, kernel_size=5, activation='relu'),
#     MaxPool1D(pool_size=2),
#     Dropout(0.2),

#     Conv1D(filters=128, kernel_size=5, activation='relu'),
#     MaxPool1D(pool_size=2),
#     Dropout(0.2),

#     Flatten(),

#     Dense(64, activation='relu'),
#     Dropout(0.2),

#     Dense(32, activation='relu'),
#     Dense(1)]
# )

# modelC.compile(
#     optimizer=Adam(learning_rate=0.0005),
#     loss="mse",
#     metrics=["mse"]
# )

# modelC.fit(
#     XLTrain,
#     YLTrain,
#     epochs=50,
#     batch_size=32,
#     validation_split = 0.1,
#     callbacks = call_back
# )

Epoch 1/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.0387 - val_loss: 0.0019
Epoch 2/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.0038 - val_loss: 0.0022
Epoch 3/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.0039 - val_loss: 0.0023
Epoch 4/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0037 - val_loss: 0.0022
Epoch 5/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.0036 - val_loss: 0.0023
Epoch 6/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0035 - val_loss: 0.0022
Epoch 7/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.0036 - val_loss: 0.0022
Epoch 8/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.0038 - val_loss: 0.0022
Epoch 9/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0035 - val_loss: 0.0022


In [66]:
Ypred = modelL.predict(XLTest)
pred = y_scaler.inverse_transform(Ypred)
print("r2score",r2_score(YLTest, pred))
print("mae",mean_absolute_error(YLTest, pred))

print("Prediction mean and std", pred.mean(), pred.std())
print("Test mean and std", YLTest.mean(), YLTest.std())

30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
r2score 0.018675690943366408
mae 0.002872421148790557
Prediction mean and std 0.007801933 0.001257552
Test mean and std 0.007059997129365867 0.004262675014094219
